 # Synth data exps


In [ ]:
#from calendar import day_abbr

#from pandas.conftest import axis

import src.config as cfg
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("white")
import numpy as np
from itertools import product
from scipy.stats import spearmanr, pearsonr, wilcoxon
from scipy.stats import ttest_rel

In [ ]:
def save_plot(plt, path, formats=None, tight=True):
    if formats is None:
        formats = ["png", "eps"]

    for fmt in formats:
        plt.savefig(
            f"{path}.{fmt}",
            dpi=300 if fmt == "png" else None,
            bbox_inches="tight" if tight else None
        )

def explode_metrics(d):
    return {key: d.get(key) for key in d.keys()}

def extract_metrics(df, metrics_to_explode, cols_to_explode_prefx):
    for col, metric in product(cols_to_explode_prefx, metrics_to_explode):
        column = "{}_{}".format(col, metric)
        extraction = df[column].apply(explode_metrics).apply(pd.Series).add_prefix(col+"_")
        df = pd.concat([df, extraction], axis=1)
    return df

def get_shap_things(d = "historical_biased_y", bias = "no_bias", s = "A", flv = "high", debiasing = "prl", discr = "xgb", n_samples=1000, verbose=True):

    probing_set_path = f"../out/data/{d}_probing-set-{bias}.preprocessed"
    shap_path = f"../out/models/shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_{n_samples}_prob_{bias}.model"
    discriminator_path = f"../out/models/discr_{discr}_{debiasing}_{flv}-{s}-{d}_{n_samples}_prob_{bias}.model_seed42"

    if verbose:
        print(shap_path)

    # load probing set and then shap values
    X_probing, Y_probing = pd.read_pickle(probing_set_path)

    with open(shap_path, "rb") as file:
        artifacts = pickle.load(file)

    #with open(discriminator_path, "rb") as file: discriminator = pickle.load(file)
    return artifacts, X_probing#, discriminator


In [ ]:
import os
os.chdir("path/src")

# Results

In [ ]:
d = "historically_biased_y"
# tipo [100, 500, 1000, 5000, 10000]
n_samples = 1000
results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))
results.head()

# Heatmaps


In [ ]:
samples = [500, 1000, 5000, 10000]
synth_datasets = cfg.probing_sets_dict.keys()

#samples = [100]
#synth_datasets = ['measurement_biased_y']

biases = list(cfg.probing_sets_dict.keys())
features = list(pd.read_pickle("../rsc/datasets/historically_biased_y.preprocessed")[0].columns)
s = 'A'
deb_dict = {
    "to": 'ThresholdOptimizer',
    "eg":'ExponentiatedGradient',
    "prl": 'PrototypeRepresentationLearner'
}
flvs = ["low", "medium", "high"]
debiasings = ["to", "eg", "prl"]
discr = "xgb"

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

bias_labels = {
    "no_bias": "No Bias",
    "historically_biased_y": "Hist-Y",
    "measurement_biased_y": "Meas-Y",
    "historically_biased_r": "Hist-R",
    "historically_biased_q": "Hist-Q",
}

from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cmap = LinearSegmentedColormap.from_list(
    "custom_diverging",
    ["#08306B", "#FFFFFF", "#67000D"]
)

for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(
            cfg.results_path.format(d + f"_{n_samples}")
        )

        for flv in flvs:
            for deb in debiasings:
                hm_df = pd.DataFrame(index=features)

                for b in biases:
                    artifacts, probing_set = get_shap_things(
                        d=d,
                        bias=b,
                        s=s,
                        flv=flv,
                        debiasing=deb,
                        discr=discr,
                        verbose=False,
                        n_samples=n_samples
                    )

                    shap_values = pd.DataFrame(
                        artifacts["shap_values"],
                        columns=features
                    )

                    matching_results = results[
                        (results["debiasing_type"] == deb_dict[deb])
                        & (results["probing_set"] == b)
                        & (results["debiasing_lv"] == flv)
                    ]

                    if len(matching_results) != 1:
                        raise ValueError(
                            f"Attesa una riga di risultati per "
                            f"{d=}, {n_samples=}, {deb=}, {flv=}, {b=}; "
                            f"trovate {len(matching_results)}."
                        )

                    y_prime = np.asarray(
                        matching_results.iloc[0]["y_prime"]
                    ).reshape(-1)

                    if len(y_prime) != len(shap_values):
                        raise ValueError(
                            f"Lunghezze diverse tra SHAP "
                            f"({len(shap_values)}) e y_prime "
                            f"({len(y_prime)}) per {d=}, {b=}."
                        )

                    # Tiene le istanze corrette
                    shap_values = shap_values.loc[y_prime == 1].copy()

                    # Normalizza gli SHAP per questo probing set
                    values = shap_values.to_numpy(dtype=float)
                    scale = np.nanmax(np.abs(values)) if values.size else 0

                    if scale > 0:
                        values = values / scale
                    else:
                        values = np.zeros_like(values)

                    shap_values = pd.DataFrame(values, columns=features)
                    mean_shap_values = shap_values.mean(axis=0).to_frame(name=b)
                    hm_df = pd.concat([hm_df, mean_shap_values], axis=1)

                plot_df = hm_df.transpose()

                # Abbrevia le label Y
                plot_df.index = [
                    bias_labels.get(
                        "-".join(map(str, label))
                        if isinstance(label, tuple)
                        else str(label),
                        str(label)
                    )
                    for label in plot_df.index
                ]

                n_rows, n_cols = plot_df.shape
                fig_size = (3, 3)

                base_path = (
                    f"../out/figures/heatmaps/"
                    f"{d}-{n_samples}-{deb}-{flv}-{s}"
                )

                # Heatmap principale
                plt.figure(figsize=fig_size)
                sns.heatmap(
                    plot_df,
                    annot=True,
                    fmt=".1f",
                    annot_kws={"fontsize": 12, "fontweight": "bold"},
                    vmin=-1,
                    vmax=1,
                    cmap=cmap,
                    cbar=False,
                    yticklabels=False
                )
                plt.xticks(fontsize=12, fontweight="bold")
                plt.xlabel("")
                plt.ylabel("")
                plt.subplots_adjust(
                    left=0.02,
                    right=0.99,
                    top=0.90,
                    bottom=0.08
                )
                save_plot(plt, base_path, tight=None)
                plt.close()

                # Label Y separate: un solo file, sovrascritto a ogni iterazione
                y_fontsize = 14
                font_prop = FontProperties(size=y_fontsize, weight="bold")

                max_text_width = max(
                    TextPath(
                        (0, 0),
                        str(label),
                        prop=font_prop
                    ).get_extents().width / 72
                    for label in plot_df.index
                )
                ylabel_width = max_text_width + 0.30

                plt.figure(figsize=(ylabel_width, fig_size[1]))
                plt.ylim(n_rows, 0)
                plt.yticks(
                    np.arange(n_rows) + 0.5,
                    plot_df.index,
                    rotation=0,
                    fontsize=y_fontsize,
                    fontweight="bold"
                )
                plt.xticks([])
                plt.tick_params(axis="y", length=0, pad=4)
                plt.gca().set_frame_on(False)

                left = (ylabel_width - 0.10) / ylabel_width
                plt.subplots_adjust(
                    left=left,
                    right=0.99,
                    top=0.90,
                    bottom=0.08
                )
                save_plot(
                    plt,
                    "../out/figures/heatmaps/ylabels",
                    tight=None
                )
                plt.close()

                # Colorbar separata: un solo file, sovrascritto a ogni iterazione
                plt.figure(figsize=(1.4, fig_size[1]))
                plt.gca().set_position([0.05, 0.08, 0.18, 0.82])

                colorbar = plt.colorbar(
                    ScalarMappable(
                        norm=Normalize(vmin=-1, vmax=1),
                        cmap=cmap
                    ),
                    cax=plt.gca(),
                    ticks=[-1, -0.5, 0, 0.5, 1]
                )

                colorbar.outline.set_visible(False)
                colorbar.ax.tick_params(labelsize=12)

                for label in colorbar.ax.get_yticklabels():
                    label.set_fontweight("bold")

                save_plot(
                    plt,
                    "../out/figures/heatmaps/colorbar",
                    tight=None
                )
                plt.close()

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))

        for group in [0, 1]:
            group_name = "protected" if group == 1 else "unprotected"

            for flv in flvs:
                for deb in debiasings:
                    hm_df = pd.DataFrame(index=features)

                    for b in biases:
                        artifacts, probing_set = get_shap_things(
                            d=d,
                            bias=b,
                            s=s,
                            flv=flv,
                            debiasing=deb,
                            discr=discr,
                            verbose=False,
                            n_samples=n_samples
                        )

                        matching_results = results[
                            (results["debiasing_type"] == deb_dict[deb])
                            & (results["probing_set"] == b)
                            & (results["debiasing_lv"] == flv)
                        ]

                        if len(matching_results) != 1:
                            raise ValueError(
                                f"Attesa una riga per {d=}, {n_samples=}, "
                                f"{deb=}, {flv=}, {b=}; trovate {len(matching_results)}."
                            )

                        y_prime = np.asarray(
                            matching_results.iloc[0]["y_prime"]
                        ).reshape(-1)

                        shap_array = np.asarray(artifacts["shap_values"])
                        protected_mask = (probing_set[s].to_numpy() == group)

                        if len(y_prime) != len(shap_array) or len(protected_mask) != len(shap_array):
                            raise ValueError(
                                f"Lunghezze diverse tra SHAP, y_prime e gruppo "
                                f"per {d=}, {b=}, {group=}."
                            )

                        corrected_mask = (y_prime == 1)
                        mask = corrected_mask & protected_mask

                        shap_values = pd.DataFrame(
                            shap_array[mask],
                            columns=features
                        )

                        values = shap_values.to_numpy(dtype=float)
                        scale = np.nanmax(np.abs(values)) if values.size else 0

                        if scale > 0:
                            values = values / scale
                        else:
                            values = np.zeros_like(values)

                        shap_values = pd.DataFrame(values, columns=features)
                        mean_shap_values = shap_values.mean(axis=0).to_frame(name=b)
                        hm_df = pd.concat([hm_df, mean_shap_values], axis=1)

                    plot_df = hm_df.transpose()

                    # Converte, ad esempio, ('to', 'low') in 'to-low'
                    plot_df.index = [
                        "-".join(map(str, label)) if isinstance(label, tuple) else str(label)
                        for label in plot_df.index
                    ]

                    n_rows, n_cols = plot_df.shape
                    fig_size = (3, 3)
                    """title = (
                        f"{d}-{n_samples}-{deb}-{flv}-{s} - "
                        f"Group {group_name.capitalize()}"
                    )"""
                    base_path = (
                        f"../out/figures/heatmaps/"
                        f"{d}-{n_samples}-{deb}-{flv}-{s}_{group_name}"
                    )

                    # Heatmap senza label Y e senza colorbar
                    plt.figure(figsize=fig_size)
                    sns.heatmap(
                        plot_df,
                        annot=True,
                        fmt=".1f",
                        annot_kws={"fontsize": 12, "fontweight": "bold"},
                        vmin=-1,
                        vmax=1,
                        cmap=cmap,
                        cbar=False,
                        yticklabels=False
                    )
                    #plt.title(title, fontsize=14, fontweight="bold")
                    plt.xticks(fontsize=12, fontweight="bold")
                    plt.xlabel("")
                    plt.ylabel("")
                    plt.subplots_adjust(
                        left=0.02, right=0.99, top=0.90, bottom=0.08
                    )
                    save_plot(plt, base_path, tight="tight")
                    plt.close()

                    # Label Y: larghezza sul testo, stessa altezza della heatmap
                    font_prop = FontProperties(size=12, weight="bold")
                    max_text_width = max(
                        TextPath(
                            (0, 0),
                            str(label),
                            prop=font_prop
                        ).get_extents().width / 72
                        for label in plot_df.index
                    )
                    ylabel_width = max_text_width + 0.20

                    plt.figure(figsize=(ylabel_width, fig_size[1]))
                    plt.ylim(n_rows, 0)
                    plt.yticks(
                        np.arange(n_rows) + 0.5,
                        plot_df.index,
                        rotation=0,
                        fontsize=12,
                        fontweight="bold"
                    )
                    plt.xticks([])
                    plt.tick_params(axis="y", length=0)
                    plt.gca().set_frame_on(False)

                    left = (ylabel_width - 0.08) / ylabel_width
                    plt.subplots_adjust(
                        left=left, right=0.99, top=0.90, bottom=0.08
                    )
                    save_plot(plt, f"../out/figures/heatmaps/ylabels", tight=None)
                    plt.close()

                    # Colorbar separata
                    plt.figure(figsize=(1.4, fig_size[1]))
                    plt.gca().set_position([0.05, 0.02, 0.18, 0.96])
                    colorbar = plt.colorbar(
                        ScalarMappable(
                            norm=Normalize(vmin=-1, vmax=1),
                            cmap=cmap
                        ),
                        cax=plt.gca(),
                        ticks=[-1, -0.5, 0, 0.5, 1]
                    )
                    colorbar.outline.set_visible(False)
                    colorbar.ax.tick_params(labelsize=12)

                    for label in colorbar.ax.get_yticklabels():
                        label.set_fontweight("bold")

                    save_plot(plt, f"c", tight=None)
                    plt.close()

# New Dataset Stats

## Correction per debiasing type and level of fairness

In [ ]:
metrics_to_explode = ["stats"]
cols_to_explode_prefx = ["new_dataset"]

cols = [#'dataset', 'sensitive',
        'probing_set', 'debiasing_type', 'debiasing_lv', 'new_dataset_total_samples',
       'new_dataset_n_protected', 'new_dataset_n_unprotected',
       'new_dataset_n_corrected', 'new_dataset_protected_corrected',
       'new_dataset_unprotected_corrected']

#expl_df = extract_metrics(results, metrics_to_explode, cols_to_explode_prefx)[cols]

"""
for b in biases:
    probing_set_df = expl_df[expl_df["probing_set"] == b]

    sns.barplot(
        x="debiasing_type",
        y="new_dataset_n_corrected",
        hue="debiasing_lv",
        data=probing_set_df,
        palette="pastel"
    )

    plt.title(f"Correction D:{d}_{n_samples} - P: {b}")
    plt.ylabel("No. of Samples Corrected")
    plt.xlabel("Debiasing Type")
    plt.legend(
        title="Debiasing Level",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=3
    )
    plt.tight_layout()

    plt.show()
"""
"""handles, labels = g.axes[0, 0].get_legend_handles_labels()
                    fig_leg = plt.figure(figsize=(6, 1))
                    fig_leg.legend(handles, labels, loc="center", ncol=3, frameon=False)"""


## Groups correction per debiasing type and level of fairness

In [ ]:
components = [
    "new_dataset_protected_corrected",
    "new_dataset_unprotected_corrected"
]

type_order = [
    "ThresholdOptimizer",
    "ExponentiatedGradient",
    "PrototypeRepresentationLearner"
]
level_order = ["low", "medium", "high"]

type_short = {
    "ThresholdOptimizer": "TO",
    "ExponentiatedGradient": "EG",
    "PrototypeRepresentationLearner": "PRL"
}
level_short = {"low": "L", "medium": "M", "high": "H"}

group_gap = 0.6  # aumenta questo valore per allargare lo spazio tra i tipi
for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))
        expl_df = extract_metrics(results, metrics_to_explode, cols_to_explode_prefx)
        for b in biases:
            data = expl_df.loc[expl_df["probing_set"] == b].copy()

            # Somma eventuali righe duplicate per tipo e livello
            df = (
                data.groupby(["debiasing_type", "debiasing_lv"], observed=True)[components]
                .sum()
                .reindex(
                    pd.MultiIndex.from_product(
                        [type_order, level_order],
                        names=["debiasing_type", "debiasing_lv"]
                    )
                )
                .dropna(how="all")
            )

            # Posizioni: livelli vicini dentro ogni tipo, spazio tra i tipi
            positions = []
            labels = []

            for type_index, tipo in enumerate(type_order):
                for level_index, livello in enumerate(level_order):
                    if (tipo, livello) in df.index:
                        positions.append(type_index * (len(level_order) + group_gap) + level_index)
                        labels.append(f"{type_short[tipo]} - {level_short[livello]}")

            plot_df = df.loc[
                [(tipo, livello) for tipo in type_order for livello in level_order
                 if (tipo, livello) in df.index]
            ]

            fig, ax = plt.subplots(figsize=(12, 6))

            protected = plot_df[components[0]].to_numpy()
            unprotected = plot_df[components[1]].to_numpy()

            ax.bar(positions, protected, color="red", label="Protected corrected")
            ax.bar(
                positions,
                unprotected,
                bottom=protected,
                color="green",
                label="Unprotected corrected"
            )

            for x, total in zip(positions, protected + unprotected):
                ax.text(x, total, f"{total:.0f}", ha="center", va="bottom")

            # Etichetta del tipo centrata sotto ciascun gruppo
            centers = [
                type_index * (len(level_order) + group_gap) + (len(level_order) - 1) / 2
                for type_index, tipo in enumerate(type_order)
                if any((tipo, livello) in plot_df.index for livello in level_order)
            ]
            present_types = [
                tipo for tipo in type_order
                if any((tipo, livello) in plot_df.index for livello in level_order)
            ]

            ax.set_xticks(centers)
            ax.set_xticklabels([type_short[tipo] for tipo in present_types])
            #ax.set_xlabel("Debiasing Type (levels: L, M, H)")
            ax.set_ylabel("Samples Corrected")
            ax.set_title(f"D:{d}_{n_samples} - P: {b}")
            ax.legend()

            # Indica il livello sotto ogni barra
            for x, label in zip(positions, labels):
                ax.annotate(
                    label.split(" - ")[1],
                    (x, 0),
                    xytext=(0, -18),
                    textcoords="offset points",
                    ha="center",
                    va="top"
                )

            fig.tight_layout()
            save_plot(plt, f"../out/figures/label_gen_stats/{d}_{n_samples}-groups-{b}")
            plt.close()

In [ ]:
components = [
    "new_dataset_protected_corrected",
    "new_dataset_unprotected_corrected",
    "new_dataset_n_protected",
    "new_dataset_n_unprotected"
]

type_order = [
    "ThresholdOptimizer",
    "ExponentiatedGradient",
    "PrototypeRepresentationLearner"
]
level_order = ["low", "medium", "high"]

type_short = {
    "ThresholdOptimizer": "TO",
    "ExponentiatedGradient": "EG",
    "PrototypeRepresentationLearner": "PRL"
}
level_short = {"low": "L", "medium": "M", "high": "H"}

group_gap = 0.8

for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(
            cfg.results_path.format(d + f"_{n_samples}")
        )
        expl_df = extract_metrics(
            results,
            metrics_to_explode,
            cols_to_explode_prefx
        )

        for b in biases:
            data = expl_df.loc[
                expl_df["probing_set"] == b
            ].copy()

            # Somma i conteggi e i denominatori di eventuali righe duplicate
            counts = (
                data.groupby(
                    ["debiasing_type", "debiasing_lv"],
                    observed=True
                )[components]
                .sum()
            )

            counts["protected_pct"] = (
                counts["new_dataset_protected_corrected"]
                .div(counts["new_dataset_n_protected"])
                .mul(100)
                .where(counts["new_dataset_n_protected"] > 0)
            )

            counts["unprotected_pct"] = (
                counts["new_dataset_unprotected_corrected"]
                .div(counts["new_dataset_n_unprotected"])
                .mul(100)
                .where(counts["new_dataset_n_unprotected"] > 0)
            )

            plot_df = (
                counts[["protected_pct", "unprotected_pct"]]
                .reindex(
                    pd.MultiIndex.from_product(
                        [type_order, level_order],
                        names=["debiasing_type", "debiasing_lv"]
                    )
                )
                .dropna(how="all")
            )

            # Posizioni e label: livelli vicini dentro ogni metodo
            positions = []
            labels = []
            protected_values = []
            unprotected_values = []

            for type_i, tipo in enumerate(type_order):
                for level_i, livello in enumerate(level_order):
                    key = (tipo, livello)

                    if key not in plot_df.index:
                        continue

                    positions.append(
                        type_i * (len(level_order) + group_gap) + level_i
                    )
                    labels.append(
                        f"{type_short[tipo]}-{level_short[livello]}"
                    )
                    protected_values.append(
                        plot_df.loc[key, "protected_pct"]
                    )
                    unprotected_values.append(
                        plot_df.loc[key, "unprotected_pct"]
                    )

            positions = np.asarray(positions)
            protected_values = np.asarray(protected_values, dtype=float)
            unprotected_values = np.asarray(unprotected_values, dtype=float)

            fig, ax = plt.subplots(figsize=(11, 6))

            # Linea che collega i due gruppi per ogni setting
            for y, protected, unprotected in zip(
                positions,
                protected_values,
                unprotected_values
            ):
                if np.isfinite(protected) and np.isfinite(unprotected):
                    ax.plot(
                        [protected, unprotected],
                        [y, y],
                        color="0.65",
                        linewidth=1.5,
                        zorder=1
                    )

            ax.scatter(
                protected_values,
                positions,
                color="red",
                s=65,
                label="Protected",
                zorder=2
            )
            ax.scatter(
                unprotected_values,
                positions,
                color="green",
                s=65,
                label="Unprotected",
                zorder=2
            )

            # Percentuali accanto ai punti
            for y, protected, unprotected in zip(
                positions,
                protected_values,
                unprotected_values
            ):
                if np.isfinite(protected):
                    ax.annotate(
                        f"{protected:.1f}%",
                        (protected, y),
                        xytext=(0, 7),
                        textcoords="offset points",
                        ha="center",
                        fontsize=8,
                        color="red"
                    )

                if np.isfinite(unprotected):
                    ax.annotate(
                        f"{unprotected:.1f}%",
                        (unprotected, y),
                        xytext=(0, -12),
                        textcoords="offset points",
                        ha="center",
                        fontsize=8,
                        color="green"
                    )

            ax.set_yticks(positions)
            ax.set_yticklabels(labels)
            ax.invert_yaxis()
            ax.set_xlim(0, 100)
            ax.set_xlabel("Percentuale di campioni corretti nel gruppo")
            ax.set_ylabel("Debiasing type e livello")
            ax.set_title(f"D:{d}_{n_samples} - P: {b}")
            ax.grid(axis="x", alpha=0.25)
            ax.legend()

            fig.tight_layout()
            save_plot(
                fig,
                f"../out/figures/label_gen_stats/"
                f"{d}_{n_samples}-groups-perc-dumbbell-{b}"
            )
            plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch

type_order = [
    "ThresholdOptimizer",
    "ExponentiatedGradient",
    "PrototypeRepresentationLearner"
]
level_order = ["low", "medium", "high"]

type_short = {
    "ThresholdOptimizer": "TO",
    "ExponentiatedGradient": "EG",
    "PrototypeRepresentationLearner": "PRL"
}
level_short = {"low": "L", "medium": "M", "high": "H"}

bias_short = {
    "no_bias": "No Bias",
    "historically_biased_y": "Hist-Y",
    "measurement_biased_y": "Meas-Y",
    "historically_biased_r": "Hist-R",
    "historically_biased_q": "Hist-Q"
}

count_columns = [
    "new_dataset_protected_corrected",
    "new_dataset_unprotected_corrected",
    "new_dataset_n_protected",
    "new_dataset_n_unprotected"
]

protected_color = "#3B6FB6"
unprotected_color = "#E68645"
bar_width = 0.40

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.labelcolor": "#333333",
    "xtick.color": "#333333",
    "ytick.color": "#333333"
})

for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(
            cfg.results_path.format(d + f"_{n_samples}")
        )
        expl_df = extract_metrics(
            results,
            metrics_to_explode,
            cols_to_explode_prefx
        )

        for tipo in type_order:
            for livello in level_order:
                data = expl_df[
                    (expl_df["debiasing_type"] == tipo)
                    & (expl_df["debiasing_lv"] == livello)
                ]

                grouped = (
                    data.groupby("probing_set", observed=True)[count_columns]
                    .sum()
                    .reindex(biases)
                )

                protected_pct = (
                    grouped["new_dataset_protected_corrected"]
                    .div(grouped["new_dataset_n_protected"])
                    .mul(100)
                    .where(grouped["new_dataset_n_protected"] > 0)
                )
                unprotected_pct = (
                    grouped["new_dataset_unprotected_corrected"]
                    .div(grouped["new_dataset_n_unprotected"])
                    .mul(100)
                    .where(grouped["new_dataset_n_unprotected"] > 0)
                )

                x = np.arange(len(biases))
                fig, ax = plt.subplots(figsize=(5, 5), facecolor="white")
                ax.set_facecolor("#FAFAFC")

                bars_protected = ax.bar(
                    x - bar_width / 2,
                    protected_pct,
                    width=bar_width,
                    color=protected_color,
                    edgecolor="white",
                    linewidth=1.2,
                    zorder=3
                )
                bars_unprotected = ax.bar(
                    x + bar_width / 2,
                    unprotected_pct,
                    width=bar_width,
                    color=unprotected_color,
                    edgecolor="white",
                    linewidth=1.2,
                    zorder=3
                )

                # Barre arrotondate
                for bars, color in [
                    (bars_protected, protected_color),
                    (bars_unprotected, unprotected_color)
                ]:
                    for bar in bars:
                        x_bar = bar.get_x()
                        y_bar = bar.get_y()
                        width = bar.get_width()
                        height = bar.get_height()

                        bar.remove()
                        if np.isfinite(height) and height > 0:
                            rounded_bar = FancyBboxPatch(
                                (x_bar, y_bar),
                                width,
                                height,
                                boxstyle="round,pad=0,rounding_size=0.08",
                                facecolor=color,
                                edgecolor="white",
                                linewidth=1.2,
                                clip_on=True,
                                zorder=3
                            )
                            ax.add_patch(rounded_bar)

                # Etichette percentuali dentro le barre: disattivate
                # for bar, value in zip(bars_protected, protected_pct):
                #     if np.isfinite(value) and value > 0:
                #         ax.text(
                #             bar.get_x() + bar.get_width() / 2,
                #             value / 2,
                #             f"{value:.1f}%",
                #             ha="center",
                #             va="center",
                #             rotation=90,
                #             fontsize=9,
                #             fontweight="bold",
                #             color="white",
                #             zorder=4
                #         )
                #
                # for bar, value in zip(bars_unprotected, unprotected_pct):
                #     if np.isfinite(value) and value > 0:
                #         ax.text(
                #             bar.get_x() + bar.get_width() / 2,
                #             value / 2,
                #             f"{value:.1f}%",
                #             ha="center",
                #             va="center",
                #             rotation=90,
                #             fontsize=9,
                #             fontweight="bold",
                #             color="white",
                #             zorder=4
                #         )

                ax.set_xticks(x)
                ax.set_xticklabels(
                    [bias_short.get(b, b) for b in biases],
                    rotation=0,
                    ha="center",
                    fontsize=13,
                    fontweight="bold"
                )

                ax.tick_params(axis="y", labelsize=12, length=0)
                for label in ax.get_yticklabels():
                    label.set_fontweight("bold")

                ax.set_xlabel("")
                ax.set_ylabel("")
                ax.set_title("")
                ax.set_ylim(0, 100)
                ax.set_xlim(-0.6, len(biases) - 0.4)

                ax.set_axisbelow(True)
                ax.grid(
                    axis="y",
                    color="#D9DDE5",
                    linestyle="-",
                    linewidth=0.8,
                    alpha=0.8,
                    zorder=0
                )

                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                ax.spines["left"].set_visible(False)
                ax.spines["bottom"].set_color("#B8BEC9")
                ax.spines["bottom"].set_linewidth(0.8)

                fig.tight_layout()
                save_plot(
                    fig,
                    f"../out/figures/label_gen_stats/"
                    f"{d}_{n_samples}-groups-perc-"
                    f"{type_short[tipo]}-{level_short[livello]}"
                )
                plt.close(fig)

fig_legend, ax_legend = plt.subplots(figsize=(2.8, 0.7))
ax_legend.axis("off")

handles = [
    Patch(facecolor=protected_color, label="Protected"),
    Patch(facecolor=unprotected_color, label="Unprotected")
]

ax_legend.legend(
    handles=handles,
    loc="center",
    ncol=2,
    frameon=False,
    prop={"weight": "bold", "size": 11}
)

fig_legend.tight_layout()
save_plot(
    fig_legend,
    "../out/figures/label_gen_stats/groups-perc-legend"
)
plt.close(fig_legend)

## Group Debiasing Correction w.r.t. probing sets

In [ ]:
for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))
        expl_df = extract_metrics(results, metrics_to_explode, cols_to_explode_prefx)
        for type_to_plot in [deb_dict[k] for k in debiasings]:
            for level_to_plot in flvs:
                components = [
                    "new_dataset_protected_corrected",
                    "new_dataset_unprotected_corrected"
                ]

                type_short = {
                    "ThresholdOptimizer": "TO",
                    "ExponentiatedGradient": "EG",
                    "PrototypeRepresentationLearner": "PRL"
                }
                level_short = {"low": "L", "medium": "M", "high": "H"}

                data = expl_df.loc[
                    (expl_df["debiasing_type"] == type_to_plot)
                    & (expl_df["debiasing_lv"] == level_to_plot)
                ].copy()

                # Aggrega eventuali righe multiple per probing set
                plot_df = (
                    data.groupby("probing_set", observed=True)[components]
                    .sum()
                    .reset_index()
                )

                labels = [
                    f"{type_short[type_to_plot]} - {level_short[level_to_plot]} - {probing_set}"
                    for probing_set in plot_df["probing_set"]
                ]

                protected = plot_df[components[0]]
                unprotected = plot_df[components[1]]
                x = range(len(plot_df))

                fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 1.2), 6))
                ax.bar(x, protected, color="red", label="Protected corrected")
                ax.bar(
                    x,
                    unprotected,
                    bottom=protected,
                    color="green",
                    label="Unprotected corrected"
                )

                for i, total in enumerate(protected + unprotected):
                    ax.text(i, total, f"{total:.0f}", ha="center", va="bottom")

                ax.set_xticks(list(x))
                ax.set_xticklabels(labels, rotation=45, ha="right")
                ax.set_ylabel("Samples Corrected")
                ax.set_xlabel("Debiasing type - level - probing set")
                ax.set_title(
                    f"D:{d}_{n_samples} - {type_short[type_to_plot]} "
                    f"(level: {level_short[level_to_plot]}) by probing set"
                )
                ax.legend()

                fig.tight_layout()
                save_plot(fig, f"../out/figures/label_gen_stats/{d}_{n_samples}-groups-{type_to_plot}-{level_to_plot}")
                plt.close()

## Groups % correction

In [ ]:
type_order = [
    "ThresholdOptimizer",
    "ExponentiatedGradient",
    "PrototypeRepresentationLearner"
]
level_order = ["low", "medium", "high"]

type_short = {
    "ThresholdOptimizer": "TO",
    "ExponentiatedGradient": "EG",
    "PrototypeRepresentationLearner": "PRL"
}
level_short = {"low": "L", "medium": "M", "high": "H"}

group_gap = 0.8  # spazio tra i diversi debiasing type
bar_width = 0.35

for d in synth_datasets:
    for n_samples in samples:
        results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))
        expl_df = extract_metrics(results, metrics_to_explode, cols_to_explode_prefx)
        for b in biases:
            data = expl_df.loc[expl_df["probing_set"] == b].copy()

            # Percentuale corretta sul totale di ciascun gruppo
            data["protected_correction_pct"] = (
                data["new_dataset_protected_corrected"]
                / data["new_dataset_n_protected"] * 100
            )
            data["unprotected_correction_pct"] = (
                data["new_dataset_unprotected_corrected"]
                / data["new_dataset_n_unprotected"] * 100
            )

            # Indice completo: mantiene anche l'ordine desiderato
            df = (
                data.set_index(["debiasing_type", "debiasing_lv"])[
                    ["protected_correction_pct", "unprotected_correction_pct"]
                ]
                .reindex(pd.MultiIndex.from_product(
                    [type_order, level_order],
                    names=["debiasing_type", "debiasing_lv"]
                ))
                .dropna(how="all")
            )

            # Posizioni delle coppie: livelli vicini dentro ogni tipo,
            # spazio aggiuntivo tra i tipi.
            positions = []
            labels = []
            values = []

            for type_i, tipo in enumerate(type_order):
                for level_i, livello in enumerate(level_order):
                    key = (tipo, livello)
                    if key in df.index:
                        x = type_i * (len(level_order) + group_gap) + level_i
                        positions.append(x)
                        labels.append(f"{type_short[tipo]}-{level_short[livello]}")
                        values.append(df.loc[key].to_numpy())

            values = np.asarray(values)
            x = np.asarray(positions)

            fig, ax = plt.subplots(figsize=(12, 6))

            protected = values[:, 0]
            unprotected = values[:, 1]

            bars_protected = ax.bar(
                x - bar_width / 2, protected, bar_width,
                color="red", label="Protected"
            )
            bars_unprotected = ax.bar(
                x + bar_width / 2, unprotected, bar_width,
                color="green", label="Unprotected"
            )

            ax.bar_label(bars_protected, fmt="%.1f%%", padding=3, fontsize=8)
            ax.bar_label(bars_unprotected, fmt="%.1f%%", padding=3, fontsize=8)

            ax.set_xticks(x)
            ax.set_xticklabels(labels)
            ax.set_ylim(0, 110)
            ax.set_ylabel("Percentuale di campioni corretti")
            ax.set_xlabel("Debiasing type e livello")
            ax.set_title(f"D:{d}_{n_samples} - P: {b}")
            ax.legend()

            fig.tight_layout()
            save_plot(fig, f"../out/figures/label_gen_stats/{d}_{n_samples}-groups-perc-{b}")
            plt.close()

# Statistical Tests

In [ ]:
feature_names = ['P', 'Q_0', 'Q_1', 'Q_2', 'Q_3', 'A']
#test_prefixes = ["t", "w"]
test_prefixes = ["t"]
test_columns = [f"{pref}_{feature}" for pref, feature in product(test_prefixes,feature_names)]
alpha = 0.05
#test_passed_df = pd.DataFrame(columns=["dataset", "n_samples", "probing_set", "debiasing_type", "debiasing_lv"] + test_columns)

rows = []
for d in synth_datasets:
    for n_samples in samples:
        #print(d, n_samples)
        results = pd.read_pickle(cfg.results_path.format(d + f"_{n_samples}"))
        for flv in flvs:
            for debiasing in debiasings:
                #print(debiasing, flv)
                for bias in biases:
                    shap_path = f"../out/models/shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_{n_samples}_prob_{bias}.model"
                    var_shap_path =f"../out/models/var_shap_{discr}_lr-{debiasing}-{flv}-{s}-{d}_{n_samples}_prob_{bias}.model"
                    temp_df = results[(results['probing_set'] == bias) & (results["debiasing_type"] ==  deb_dict[debiasing]) & (results["debiasing_lv"] == flv)]

                    if temp_df.shape[0] != 1: raise ValueError("More than one match or None.")
                    if temp_df['discriminator_var_accuracy'].iloc[0] is not None:
                        with open(shap_path, "rb") as f: shap_aid = pickle.load(f)
                        with open(var_shap_path, "rb") as f: shap_var = pickle.load(f)

                        shap_var_values = pd.DataFrame(shap_var['shap_values'], columns=shap_var['feature_names'])
                        shap_aid_values = pd.DataFrame(shap_aid['shap_values'], columns=shap_aid['feature_names'])

                        test_df = pd.DataFrame(index=shap_var_values.columns)
                        row = {
                            "dataset": d,
                            "n_samples": n_samples,
                            "probing_set": bias,
                            "debiasing_type": deb_dict[debiasing],
                            "debiasing_lv": flv
                        }
                        tests_passed_t = True
                        tests_passed_w = True
                        for f in shap_var_values.columns:
                            #pair = pd.DataFrame([shap_var_values[f].rename(f"{f}-VAR"), shap_aid_values[f].rename(f"{f}-AID")]).T
                            aid = shap_aid_values[f].to_numpy()
                            var = shap_var_values[f].to_numpy()
                            #r_pearson, p_pearson = pearsonr(aid, var) # FIXME:
                            #r_spearman, p_spearman = spearmanr(aid, var) # FIXME:
                            test = wilcoxon(aid, var, alternative="two-sided", method="auto")
                            w_stat, p_wilcoxon = test.statistic, test.pvalue
                            t_stat, p_ttest = ttest_rel(aid, var)

                            test_df.loc[f, "t_stat"] = t_stat
                            test_df.loc[f, "p_ttest"] = p_ttest
                            test_df.loc[f, 'p_wilcoxon'] = p_wilcoxon
                            test_df.loc[f, 'w_stat'] = w_stat
                            #test_df.loc[f, "r_pearson"] = r_pearson
                            #test_df.loc[f, 'p_pearson'] = p_pearson
                            #test_df.loc[f, 'r_spearman'] = r_spearman
                            #test_df.loc[f, 'p_spearman'] = p_spearman

                            row[f't_{f}'] = p_ttest < alpha
                            #row[f'w_{f}'] = p_wilcoxon < alpha
                            if p_ttest >= alpha: tests_passed_t = False
                            #if p_wilcoxon < alpha: tests_passed_w = False
                            #row['w_passed'] = tests_passed_w
                            row['t_passed'] = tests_passed_t

                        # TODO: add correction?
                        rows.append(row)
                    else:
                        rows.append({
                            "dataset": d,
                            "n_samples": n_samples,
                            "probing_set": bias,
                            "debiasing_type": deb_dict[debiasing],
                            "debiasing_lv": flv,
                            't_P': None,
                            't_Q_0': None,
                            't_Q_1': None,
                            't_Q_2': None,
                            't_Q_3': None,
                            't_A': None,
                            #'wilcoxon_P': None,
                            #'wilcoxon_Q_0': None,
                            #'wilcoxon_Q_1': None,
                            #'wilcoxon_Q_2': None,
                            #'wilcoxon_Q_3': None,
                            #'wilcoxon_A': None
                        })
        #break
    #break
test_passed_df = pd.DataFrame(rows)
#test_passed_df["tests_passed"] = test_passed_df['w_passed'] & test_passed_df['t_passed']
test_passed_df.set_index(['dataset', 'n_samples', 'debiasing_type', "debiasing_lv", "probing_set"])
test_passed_df.to_csv("../out/figures/tables/test_passed.csv")
test_passed_df.to_pickle("../out/figures/tables/test_passed.df")

In [2]:
import pandas as pd
table = pd.read_pickle("../out/figures/tables/test_passed.df")
table = table.fillna(True) # no viarance so p=0.00

In [14]:
from itertools import combinations
import pandas as pd

table = pd.read_pickle("../out/figures/tables/test_passed.df")

param_cols = [
    "dataset",
    "n_samples",
    "probing_set",
    "debiasing_type",
    "debiasing_lv",
]
test_cols = ["t_P", "t_passed", "t_Q_0", "t_Q_1", "t_Q_2", "t_Q_3", "t_A"]

data = table.copy()
data[test_cols] = data[test_cols].fillna(True)

# Per ogni coppia, colonne = combinazioni dei valori dei due parametri.
tabelle_per_coppia = {}
for coppia in combinations(param_cols, 2):
    tabelle_per_coppia[coppia] = (
        data.groupby(list(coppia), dropna=False)[test_cols]
            .mean()
            #.mul(100)
            .T
            .round(2)
    )

tabelle_per_coppia[("probing_set", "debiasing_lv")]

probing_set  historically_biased_q                     historically_biased_r  \
debiasing_lv                  high       low    medium                  high   
t_P                       0.016667  0.016667  0.016667              0.016667   
t_passed                  0.683333  0.716667      0.65                   0.8   
t_Q_0                     0.016667  0.016667  0.016667              0.016667   
t_Q_1                     0.016667  0.016667  0.016667              0.016667   
t_Q_2                     0.016667  0.016667  0.016667              0.016667   
t_Q_3                     0.016667  0.016667  0.016667              0.016667   
t_A                       0.016667  0.016667  0.016667              0.016667   

probing_set                      historically_biased_y                      \
debiasing_lv       low    medium                  high       low    medium   
t_P           0.016667  0.016667              0.016667  0.016667  0.016667   
t_passed           0.5  0.583333              0.816667  0.583333  0.616667   
t_Q_0         0.016667  0.016667              0.016667  0.016667  0.016667   
t_Q_1         0.016667  0.016667              0.016667  0.016667  0.016667   
t_Q_2         0.016667  0.016667              0.016667  0.016667  0.016667   
t_Q_3         0.016667  0.016667              0.016667  0.016667  0.016667   
t_A           0.016667  0.016667              0.016667  0.016667  0.016667   

probing_set  measurement_biased_y                       no_bias            \
debiasing_lv                 high       low    medium      high       low   
t_P                      0.016667  0.016667  0.016667  0.016667  0.016667   
t_passed                 0.716667  0.616667      0.65  0.716667  0.483333   
t_Q_0                    0.016667  0.016667  0.016667  0.016667  0.016667   
t_Q_1                    0.016667  0.016667  0.016667  0.016667  0.016667   
t_Q_2                    0.016667  0.016667  0.016667  0.016667  0.016667   
t_Q_3                    0.016667  0.016667  0.016667  0.016667  0.016667   
t_A                      0.016667  0.016667  0.016667  0.016667  0.016667   

probing_set             
debiasing_lv    medium  
t_P           0.016667  
t_passed      0.583333  
t_Q_0         0.016667  
t_Q_1         0.016667  
t_Q_2         0.016667  
t_Q_3         0.016667  
t_A           0.016667

In [39]:
import pandas as pd

table = pd.read_pickle("../out/figures/tables/test_passed.df")

param_cols = [
    "dataset",
    "n_samples",
    "probing_set",
    "debiasing_type",
    "debiasing_lv",
]
test_cols = ["t_P", "t_passed", "t_Q_0", "t_Q_1", "t_Q_2", "t_Q_3", "t_A"]

data = table.copy()
data[test_cols] = data[test_cols].fillna(True)

# Una colonna per ogni singolo valore di ogni parametro.
# Ogni valore aggrega tutti i run in cui quel parametro assume quel valore.
risultato = pd.concat(
    {
        param: (
            data.groupby(param, dropna=False)[test_cols]
                .mean()
                #.mul(100)
                .T
        )
        for param in param_cols
    },
    axis=1,
)

# Appiattisci i nomi delle colonne: "n_samples-500", ecc.
risultato.columns = [
    f"{param}-{value}" for param, value in risultato.columns
]

risultato.index.name = "feature"
risultato = risultato.round(2)

risultato

,dataset-historically_biased_q,dataset-historically_biased_r,dataset-historically_biased_y,dataset-measurement_biased_y,dataset-no_bias,n_samples-500,n_samples-1000,n_samples-5000,n_samples-10000,probing_set-historically_biased_q,probing_set-historically_biased_r,probing_set-historically_biased_y,probing_set-measurement_biased_y,probing_set-no_bias,debiasing_type-ExponentiatedGradient,debiasing_type-PrototypeRepresentationLearner,debiasing_type-ThresholdOptimizer,debiasing_lv-high,debiasing_lv-low,debiasing_lv-medium
feature,,,,,,,,,,,,,,,,,,,,
t_P,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_passed,0.833333,0.622222,0.572222,0.677778,0.533333,0.666667,0.564444,0.702222,0.657778,0.683333,0.627778,0.672222,0.661111,0.594444,0.716667,0.656667,0.57,0.746667,0.58,0.616667
t_Q_0,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_1,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_2,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_3,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_A,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333


In [40]:
param_labels = {
    "probing_set": "probing",
    "dataset": "dataset",
    "debiasing_type": "debiasing",
    "debiasing_lv": "lv",
    "n_samples": "n",
}

value_labels = {
    "probing_set": {
        "no_bias": "No-Bias",
        "historically_biased_y": "Hist-Y",
        "historically_biased_r": "Hist-R",
        "historically_biased_q": "Hist-Q",
        "measurement_biased_y": "Meas-Y",
    },
    "dataset": {
        "no_bias": "No-Bias",
        "historically_biased_y": "Hist-Y",
        "historically_biased_r": "Hist-R",
        "historically_biased_q": "Hist-Q",
        "measurement_biased_y": "Meas-Y",
    },
    "debiasing_type": {
        "ThresholdOptimizer": "TO",
        "PrototypeRepresentationLearner": "PRL",
        "ExponentiatedGradient": "EG",
    },
    "debiasing_lv": {
        "low": "L",
        "medium": "M",
        "high": "H",
    },
}

def rinomina_colonna(col):
    for param, label in param_labels.items():
        prefix = f"{param}-"
        if col.startswith(prefix):
            value = col[len(prefix):]
            value = value_labels.get(param, {}).get(value, value)
            return f"{label}-{value}"
    return col

risultato = risultato.rename(columns=rinomina_colonna)
risultato

,dataset-Hist-Q,dataset-Hist-R,dataset-Hist-Y,dataset-Meas-Y,dataset-No-Bias,n-500,n-1000,n-5000,n-10000,probing-Hist-Q,probing-Hist-R,probing-Hist-Y,probing-Meas-Y,probing-No-Bias,debiasing-EG,debiasing-PRL,debiasing-TO,lv-H,lv-L,lv-M
feature,,,,,,,,,,,,,,,,,,,,
t_P,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_passed,0.833333,0.622222,0.572222,0.677778,0.533333,0.666667,0.564444,0.702222,0.657778,0.683333,0.627778,0.672222,0.661111,0.594444,0.716667,0.656667,0.57,0.746667,0.58,0.616667
t_Q_0,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_1,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_2,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_Q_3,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333
t_A,0.005556,0.005556,0.005556,0.005556,0.005556,0.004444,0.004444,0.004444,0.004444,0.005556,0.005556,0.005556,0.005556,0.005556,0.003333,0.003333,0.003333,0.003333,0.003333,0.003333


# Heatmaps Real Datasets

In [ ]:
datasets = ["adult", "bank_marketing", "credit_card"]
old_path = "../out/real/"
feature_types_adult = {
    # Demographic attribute
    "age": "Q",

    # Workclass: employment context / socioeconomic proxy
    "workclass_Federal-gov": "P",
    "workclass_Local-gov": "P",
    "workclass_Never-worked": "P",
    "workclass_Private": "P",
    "workclass_Self-emp-inc": "P",
    "workclass_Self-emp-not-inc": "P",
    "workclass_State-gov": "P",
    "workclass_Without-pay": "P",

    # Census sampling weight
    "fnlwgt": "Q",

    # Education: human capital / income-related proxy
    "education_10th": "P",
    "education_11th": "P",
    "education_12th": "P",
    "education_1st-4th": "P",
    "education_5th-6th": "P",
    "education_7th-8th": "P",
    "education_9th": "P",
    "education_Assoc-acdm": "P",
    "education_Assoc-voc": "P",
    "education_Bachelors": "P",
    "education_Doctorate": "P",
    "education_HS-grad": "P",
    "education_Masters": "P",
    "education_Preschool": "P",
    "education_Prof-school": "P",
    "education_Some-college": "P",
    "education-num": "P",

    # Marital status: demographic and socioeconomic proxy
    "marital-status_Divorced": "P_A",
    "marital-status_Married-AF-spouse": "P_A",
    "marital-status_Married-civ-spouse": "P_A",
    "marital-status_Married-spouse-absent": "P_A",
    "marital-status_Never-married": "P_A",
    "marital-status_Separated": "P_A",
    "marital-status_Widowed": "P_A",

    # Occupation: employment and income-related proxy
    "occupation_Adm-clerical": "P",
    "occupation_Armed-Forces": "P",
    "occupation_Craft-repair": "P",
    "occupation_Exec-managerial": "P",
    "occupation_Farming-fishing": "P",
    "occupation_Handlers-cleaners": "P",
    "occupation_Machine-op-inspct": "P",
    "occupation_Other-service": "P",
    "occupation_Priv-house-serv": "P",
    "occupation_Prof-specialty": "P",
    "occupation_Protective-serv": "P",
    "occupation_Sales": "P",
    "occupation_Tech-support": "P",
    "occupation_Transport-moving": "P",

    # Relationship: potentially informative about sex
    "relationship_Husband": "P_A",
    "relationship_Not-in-family": "P_A",
    "relationship_Other-relative": "P_A",
    "relationship_Own-child": "P_A",
    "relationship_Unmarried": "P_A",
    "relationship_Wife": "P_A",

    # Sensitive attributes
    "race": "A",
    "sex": "A",

    # Direct financial resources
    "capital-gain": "R",
    "capital-loss": "R",

    # Labour input / socioeconomic proxy
    "hours-per-week": "P",

    # Country of origin: demographic/geographic proxy
    "native-country_Cambodia": "P_A",
    "native-country_Canada": "P_A",
    "native-country_China": "P_A",
    "native-country_Columbia": "P_A",
    "native-country_Cuba": "P_A",
    "native-country_Dominican-Republic": "P_A",
    "native-country_Ecuador": "P_A",
    "native-country_El-Salvador": "P_A",
    "native-country_England": "P_A",
    "native-country_France": "P_A",
    "native-country_Germany": "P_A",
    "native-country_Greece": "P_A",
    "native-country_Guatemala": "P_A",
    "native-country_Haiti": "P_A",
    "native-country_Holand-Netherlands": "P_A",
    "native-country_Honduras": "P_A",
    "native-country_Hong": "P_A",
    "native-country_Hungary": "P_A",
    "native-country_India": "P_A",
    "native-country_Iran": "P_A",
    "native-country_Ireland": "P_A",
    "native-country_Italy": "P_A",
    "native-country_Jamaica": "P_A",
    "native-country_Japan": "P_A",
    "native-country_Laos": "P_A",
    "native-country_Mexico": "P_A",
    "native-country_Nicaragua": "P_A",
    "native-country_Outlying-US(Guam-USVI-etc)": "P_A",
    "native-country_Peru": "P_A",
    "native-country_Philippines": "P_A",
    "native-country_Poland": "P_A",
    "native-country_Portugal": "P_A",
    "native-country_Puerto-Rico": "P_A",
    "native-country_Scotland": "P_A",
    "native-country_South": "P_A",
    "native-country_Taiwan": "P_A",
    "native-country_Thailand": "P_A",
    "native-country_Trinadad&Tobago": "P_A",
    "native-country_United-States": "P_A",
    "native-country_Vietnam": "P_A",
    "native-country_Yugoslavia": "P_A",
}
feature_types_credit_card = {
    # Credit limit: direct financial resource
    "limit_bal": "R",

    # Sensitive attribute
    "sex": "A",

    # Education: demographic and socioeconomic proxy
    "education_Grad.School": "P_A",
    "education_High School": "P_A",
    "education_Others": "P_A",
    "education_University": "P_A",

    # Marital status: demographic proxy
    "marital": "P_A",

    # Age: additional demographic information
    "age": "Q",

    # Repayment status: proxy for financial reliability
    "pay_0": "P",
    "pay_2": "P",
    "pay_3": "P",
    "pay_4": "P",
    "pay_5": "P",
    "pay_6": "P",

    # Bill statement amounts: financial information
    "bill_amt1": "R",
    "bill_amt2": "R",
    "bill_amt3": "R",
    "bill_amt4": "R",
    "bill_amt5": "R",
    "bill_amt6": "R",

    # Previous payment amounts: financial information
    "pay_amt1": "R",
    "pay_amt2": "R",
    "pay_amt3": "R",
    "pay_amt4": "R",
    "pay_amt5": "R",
    "pay_amt6": "R",
}
feature_types_bank_marketing = {
    # Demographic information
    "age": "Q",

    # Occupation: socioeconomic proxies
    "job_admin.": "P",
    "job_blue-collar": "P",
    "job_entrepreneur": "P",
    "job_housemaid": "P",
    "job_management": "P",
    "job_retired": "P",
    "job_self-employed": "P",
    "job_services": "P",
    "job_student": "P",
    "job_technician": "P",
    "job_unemployed": "P",

    # Marital status: demographic proxy
    "marital": "A",

    # Education: socioeconomic proxy
    "education_primary": "P",
    "education_secondary": "P",
    "education_tertiary": "P",

    # Previous financial default
    "default": "P",

    # Direct financial resource
    "balance": "R",

    # Credit-related information
    "housing": "P",
    "loan": "P",

    # Contact information
    "contact": "Q",
    "day_of_week": "Q",
    "month_apr": "Q",
    "month_aug": "Q",
    "month_dec": "Q",
    "month_feb": "Q",
    "month_jan": "Q",
    "month_jul": "Q",
    "month_jun": "Q",
    "month_mar": "Q",
    "month_may": "Q",
    "month_nov": "Q",
    "month_oct": "Q",
    "month_sep": "Q",

    # Current campaign information
    "duration": "Q",
    "campaign": "Q",

    # Previous campaign information
    "pdays": "Q",
    "previous": "Q",
    "poutcome_failure": "Q",
    "poutcome_other": "Q",
    "poutcome_success": "Q",
}

feature_types = {
    "adult": feature_types_adult,
    "bank_marketing": feature_types_bank_marketing,
    "credit_card": feature_types_credit_card,
}

def max_abs_signed(df):
    values = df.to_numpy().flatten()
    return values[np.nanargmax(np.abs(values))]

In [ ]:
hmps = []
for d in datasets:
    results_df = pd.read_pickle(old_path + f"results_{d}.df")
    sensitives = results_df['sensitive'].unique()
    with open(old_path + f"{d}_probing.data", "rb") as f:
        X_probing, y_probing = pickle.load(f)
    for s in sensitives:
        for deb in debiasings:
            for flv in flvs:
                #print(d, s, deb, flv)
                shap_path = f"shap_xgb_lr-{deb}-{flv}-{s}-{d}.model"
                with open(old_path + shap_path, "rb") as f:
                    artifacts = pickle.load(f)
                shap_values = pd.DataFrame(artifacts['shap_values'][:, :, 1], columns=X_probing.columns)


                details = results_df[(results_df['sensitive'] == s) & (results_df['debiasing_type'] == deb_dict[deb]) & (results_df['debiasing_lv'] == flv) & (results_df['discr_model_type'] == 'XGBClassifier')]
                assert len(details) == 1

                y_prime = details['y_prime'].reset_index(drop=True).iloc[0]
                corrected_mask = np.asarray(y_prime) == 1
                assert len(y_prime) == len(shap_values)
                assert len(corrected_mask) == len(shap_values)
                shap_values = shap_values[corrected_mask]
                # ----
                values = shap_values.to_numpy(dtype=float)
                scale = np.nanmax(np.abs(values))

                if scale > 0:
                    values = values / scale
                else:
                    values = np.zeros_like(values)

                shap_values = pd.DataFrame(values, columns=shap_values.columns)

                # ------
                var_type = {}

                for col in shap_values.columns:
                    tipo = feature_types[d][col]

                    if tipo not in var_type:
                        var_type[tipo] = pd.DataFrame()

                    var_type[tipo][col] = shap_values[col]

                svs_per_type = pd.DataFrame({
                    feature_type: max_abs_signed(var_type[feature_type])
                    for feature_type in var_type
                }, index=[0])

                svs_per_type.insert(0, 'dataset', d)
                svs_per_type.insert(1, 's', s)
                svs_per_type.insert(2, 'deb', deb)
                svs_per_type.insert(3, 'flv', flv)

                #svs_per_type.insert(0, 'index', f"{d}-{s}-{deb}-{flv}")

                hmps.append(svs_per_type)

hmps_df = pd.concat(hmps, ignore_index=True).set_index(['dataset', 's', 'deb', 'flv'])[['P', 'Q', 'A', 'P_A', 'R']]
hmps_df = hmps_df.fillna(0)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")

cmap = LinearSegmentedColormap.from_list(
    "custom_diverging",
    ["#08306B", "#FFFFFF", "#67000D"]
)

for d in datasets:
    df_to_plot = hmps_df.loc[d]

    if d == "adult":
        sensitives = ["sex", "race"]
    elif d == "bank_marketing":
        sensitives = ["marital"]
    elif d == "credit_card":
        sensitives = ["sex"]

    for s in sensitives:
        to_plot = df_to_plot.loc[s].copy()

        # Converte, ad esempio, ('to', 'low') in 'to-low'
        to_plot.index = [
            "-".join(map(str, label)) if isinstance(label, tuple) else str(label)
            for label in to_plot.index
        ]

        scaled = to_plot / to_plot.max().max()
        n_rows, n_cols = scaled.shape
        fig_size = (3, 3)

        # Heatmap
        plt.figure(figsize=fig_size)
        sns.heatmap(
            scaled,
            annot=True,
            fmt=".1f",
            annot_kws={"fontsize": 12, "fontweight": "bold"},
            vmin=-1,
            vmax=1,
            cmap=cmap,
            cbar=False,
            yticklabels=False
        )
        plt.title(f"{d}-{s}", fontsize=16, fontweight="bold")
        plt.xticks(fontsize=12, fontweight="bold")
        plt.xlabel("")
        plt.ylabel("")
        plt.subplots_adjust(left=0.02, right=0.99, top=0.90, bottom=0.08)
        save_plot(
            plt,
            f"../out/figures/heatmaps/{d}-{s}",
            tight=None
        )
        plt.close()

        # Label Y: larghezza quanto basta per il testo, altezza uguale alla heatmap
        font_prop = FontProperties(size=12, weight="bold")
        max_text_width = max(
            TextPath((0, 0), str(label), prop=font_prop).get_extents().width / 72
            for label in scaled.index
        )
        ylabel_width = max_text_width + 0.20

        plt.figure(figsize=(ylabel_width, fig_size[1]))
        plt.ylim(n_rows, 0)
        plt.yticks(
            np.arange(n_rows) + 0.5,
            scaled.index,
            rotation=0,
            fontsize=12,
            fontweight="bold"
        )
        plt.xticks([])
        plt.tick_params(axis="y", length=0)
        plt.gca().set_frame_on(False)

        # Colloca l'asse a destra, lasciando a sinistra lo spazio per le label
        left = (ylabel_width - 0.08) / ylabel_width
        plt.subplots_adjust(left=left, right=0.99, top=0.90, bottom=0.08)

        save_plot(
            plt,
            f"../out/figures/heatmaps/ylabels",
            tight=None
        )
        plt.close()

        # Colorbar stretta, con scala e valori in grassetto
        plt.figure(figsize=(1.4, fig_size[1]))
        plt.gca().set_position([0.05, 0.02, 0.18, 0.96])
        colorbar = plt.colorbar(
            ScalarMappable(norm=Normalize(vmin=-1, vmax=1), cmap=cmap),
            cax=plt.gca(),
            ticks=[-1, -0.5, 0, 0.5, 1]
        )
        colorbar.outline.set_visible(False)
        colorbar.ax.tick_params(labelsize=12)

        for label in colorbar.ax.get_yticklabels():
            label.set_fontweight("bold")

        save_plot(
            plt,
            f"../out/figures/heatmaps/colorbar",
            tight=None
        )
        plt.close()